In [1]:
from sklearn.datasets import fetch_covtype
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from moe_model import MoE,DenseModel
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
print(device)

cuda


In [ ]:
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Nom du GPU : {torch.cuda.get_device_name(0)}")
print(f"Mémoire totale : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} Go")
print(f"Mémoire allouée : {torch.cuda.memory_allocated(0) / 1024**3:.2f} Go")
print(f"Mémoire réservée : {torch.cuda.memory_reserved(0) / 1024**3:.2f} Go")

GPU disponible : True
Nom du GPU : NVIDIA GeForce GTX 1650
Mémoire totale : 3.81 Go
Mémoire allouée : 0.00 Go
Mémoire réservée : 0.00 Go


In [ ]:
X,y = fetch_covtype(return_X_y=True)
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=True,random_state=0)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train -=1     # Label de covtype : [1,7] nous on veut [0,6]
y_test -=1 

X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)
X_test = torch.FloatTensor(X_test).to(device)
y_test = torch.LongTensor(y_test).to(device)

epochs = 500
criterion = nn.CrossEntropyLoss()
print(X.shape)

#subset = 50_000  # ou même 50_000
#X_train, y_train = X_train[:subset], y_train[:subset]
#X_test, y_test = X_test[:subset//5], y_test[:subset//5]

batch_size = 1024
train_ds = TensorDataset(X_train,y_train)
train_loader = DataLoader(train_ds,batch_size=batch_size,shuffle=True,num_workers=2,pin_memory=True)

print(X_train.shape)
print(X_test.shape)

(581012, 54)
torch.Size([50000, 54])
torch.Size([10000, 54])


In [ ]:
from collections import defaultdict


input_dim = X_train.shape[1]
hidden_dim = 128
output_dim = len(set(y))
e_mini = 1
e_maxi = 8

ela = defaultdict(lambda:[None,None])
experts_distrib = []

    
for e in range(e_mini,e_maxi):
    model = MoE(e,input_dim,hidden_dim,output_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        for xb, yb in train_loader:
            xb,yb=xb.to(device,non_blocking=True), yb.to(device,non_blocking=True)
            optimizer.zero_grad()
            logits = model(xb)
        #print(outputs)
            loss = criterion(logits,yb)
            loss.backward()
            optimizer.step()

        #if (epoch+1) % 20 == 0:
        #    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
            ela[e][0] = loss.item()
    if model.last_counts is not None:
        experts_distrib.append(model.last_counts.cpu().numpy())
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        preds = outputs.argmax(dim=1)   
        acc = (preds == y_test).float().mean().item()
        ela[e][1] = acc
#print(f"Test accuracy: {acc:.4f}")

In [6]:
print(ela)

defaultdict(<function <lambda> at 0x7cd5f688f9c0>, {1: [0.4520167112350464, 0.8018999695777893], 2: [0.34063246846199036, 0.8326999545097351]})


In [ ]:
la = [float("-inf"),float("-inf")]
dense_model = DenseModel(input_dim,256,output_dim).to(device)
optimizer = optim.Adam(dense_model.parameters(), lr=1e-3)
dense_model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = dense_model(X_train)
    loss = criterion(outputs,y_train)
    loss.backward()
    optimizer.step()
    #if (epoch+1) % 20 == 0:
     #   print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
    if (epoch+1)%100 == 0:
        la[0] = loss.item()


In [ ]:
dense_model.eval()

with torch.no_grad():
    outputs = dense_model(X_test)
    preds = outputs.argmax(dim=1)   
    acc = (preds == y_test).float().mean().item()
    la[1] = acc
print(f"Test accuracy: {acc:.4f}")

In [ ]:
# ===============================
# Etudier l'impact && Visualiser
# ===============================
import matplotlib.pyplot as plt
import numpy as np
#1. Classification / donnée tabulaire / load_digits (1780 samples et 10 classes) petit dataset.

# Impact
# Nombre d'experts
## LOSS 
#ajouter la loss du dense model y = x
plt.figure(figsize=(8,6),facecolor="yellow")
x = np.arange(e_mini, e_maxi)
y = [ela[i][0] for i in range(e_mini, e_maxi)]

y_scaled = np.array(y) * 1e5
plt.plot(x, y_scaled, marker='o')
plt.plot(x,[la[0]*1e5]*len(x),color="red")
plt.xlabel('Number of experts')
plt.ylabel('Final loss')
plt.title('Impact of number of experts on Loss')
plt.grid(True)
plt.savefig("assets/ImpactOfExpertsOnLossCovtype.jpg",dpi=300,bbox_inches="tight")
plt.show()
## ACCURACY
# pareil pr accuracy
plt.figure(figsize=(8,6),facecolor="green")
x = np.arange(1, 6)
y = [ela[i][1] for i in range(1, 6)]
plt.plot(x, y, marker='o')
plt.plot(x,[la[1]]*len(x),color="red")

plt.xlabel('Number of experts')
plt.ylabel('Accuracy of the Moe')
plt.title('Impact of number of experts on Accuracy')
plt.grid(True)
plt.savefig("assets/ImpactOfExpertsOnAccuracyCovType.jpg",dpi=300,bbox_inches="tight")
plt.show()


In [ ]:
# Distribution en nombre de samples + annotation % dans chaque segment
plt.figure(figsize=(10,8), facecolor="gray")
n_configs = len(experts_distrib)
x = np.arange(e_mini, n_configs + 1)
max_experts = max(len(c) for c in experts_distrib)
width = 0.6
bottom = np.zeros(n_configs)
totals = np.array([c.sum() for c in experts_distrib])
cmap = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(max_experts)]

for expert_idx in range(max_experts):
    vals = np.array([c[expert_idx] if expert_idx < len(c) else 0 for c in experts_distrib])
    plt.bar(x, vals, width, bottom=bottom, color=colors[expert_idx], label=f'Expert {expert_idx+1}')
    # ajouter le pourcentage centré dans le segment si valeur > 0
    for xi, v, b, tot in zip(x, vals, bottom, totals):
        if v > 0 and tot > 0:
            pct = v / tot * 100.0
            plt.text(float(xi), b + v / 2, f"{pct:.1f}%", ha='center', va='center', color='white', fontsize=9)
    bottom += vals

plt.xlabel('Number of experts')
plt.ylabel('Number of samples assigned')
plt.title("Distribution des samples par expert selon le nombre d'experts")
plt.xticks(x)
plt.ylim(0, totals.max() * 1.05)
plt.legend(title='Experts', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("assets/DistributionOfSamplesAmongExperts_countsCovType.jpg", dpi=300, bbox_inches="tight")
plt.show()
